Opimization Exercises - Set 7 // Ali khabazian - 402416066

Q1 - MCMC

In [1]:
import numpy as np
import emcee

In [2]:
data = np.loadtxt('C:\\Users\\user\\OneDrive\\Desktop\\software\\dataprof.txt')
x = data[:, 0]
y = data[:, 1]

In [3]:
# Define the model and log-likelihood function
def log_likelihood(theta, x, y):
    a, H = theta
    y_theory = a * x**H
    sigma = y.std()  # Assuming constant standard deviation
    return -0.5 * np.sum((y - y_theory)**2 / sigma**2)

# Define the prior distributions (uniform priors in this case)
def log_prior(theta):
    a, H = theta
    if 0.1 < a < 10.0 and -5.0 < H < 5.0:
        return 0.0
    else:
        return -np.inf

# Define the log-probability function
def log_probability(theta, x, y):
    log_prior_prob = log_prior(theta)
    if not np.isfinite(log_prior_prob):
        return -np.inf
    log_likelihood_prob = log_likelihood(theta, x, y)
    return log_prior_prob + log_likelihood_prob

In [4]:
ndim = 2  
nwalkers = 32  
initial_guess = np.array([1.0, 1.0])  
initial_positions = initial_guess + 1e-4 * np.random.randn(nwalkers, ndim)

In [12]:
# Run the MCMC sampler
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(x, y))
state = sampler.run_mcmc(initial_positions, 5000, progress=True)

# Get the MCMC samples
samples = sampler.get_chain(flat=True)

# Compute the best-fit parameters and their uncertainties
a_best, H_best = np.median(samples, axis=0)
a_std, H_std = np.std(samples, axis=0)

print(f"Best-fit parameters:")
print(f"a = {a_best:.4f} +/- {a_std:.4f}")
print(f"H = {H_best:.4f} +/- {H_std:.4f}")

100%|██████████| 5000/5000 [00:05<00:00, 854.39it/s]

Best-fit parameters:
a = 1.9536 +/- 0.4951
H = 0.1064 +/- 0.1369


Q2 - HMC

In [22]:
data = np.loadtxt('C:\\Users\\user\\OneDrive\\Desktop\\software\\dataprof.txt')

x = data[:, 0]
y = data[:, 1]
yerr = data[:, 2]

In [23]:
# Define the model function
def model(x, a, H):
    return a * x ** H

# Define the log-likelihood function
def lnprob(theta, x, y, yerr):
    a, H = theta
    model_y = model(x, a, H)
    return -0.5 * np.sum(((y - model_y) / yerr) ** 2)

In [24]:
ndim = 2
nwalkers = 32
pos = [np.array([2.5, 0.5]) + 1e-4 * np.random.randn(ndim) for i in range(nwalkers)]
sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob, args=(x, y, yerr))

In [25]:
# Run the HMC sampling
sampler.run_mcmc(pos, 5000, progress=True)

# Extract the samples
samples = sampler.get_chain(discard=1000, thin=15, flat=True)
a_samples = samples[:, 0]
H_samples = samples[:, 1]

# Compute the median and 95% credible intervals
a_median = np.median(a_samples)
a_lower, a_upper = np.percentile(a_samples, [2.5, 97.5])
H_median = np.median(H_samples)
H_lower, H_upper = np.percentile(H_samples, [2.5, 97.5])

print(f"a = {a_median:.5f} (+{a_upper - a_median:.5f}, -{a_median - a_lower:.5f})")
print(f"H = {H_median:.5f} (+{H_upper - H_median:.5f}, -{H_median - H_lower:.5f})")

100%|██████████| 5000/5000 [00:03<00:00, 1420.67it/s]

a = 1.95742 (+0.03110, -0.03025)
H = 0.10653 (+0.00434, -0.00429)
